# Phase 2: Nonlinear Least-Squares Tuning Analysis

Von Mises model: r(theta) = b + a * exp(kappa * cos(2*(theta - theta0)))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC = os.path.join(BASE, 'data', 'processed')
FINAL = os.path.join(BASE, 'data', 'final')

df = pd.read_parquet(os.path.join(FINAL, 'orientation_neuron_analysis.parquet'))
bd = np.load(os.path.join(PROC, 'orientation_binned_tuning.npz'))
tuning_mean = bd['tuning_mean']
tuning_sem = bd['tuning_sem']
centers_deg = bd['angle_bin_centers_deg']

print(f'Loaded {len(df)} neurons')
df.head()


## Fit Summary

In [ ]:
summary = pd.read_csv(os.path.join(BASE, 'reports', 'tables', 'tuning_fit_summary.csv'))
summary


## Tuning Parameter Distributions

In [ ]:
df_fit = df[df['fit_success']]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df_fit['fit_r2'], bins=60, color='teal')
axes[0].set_xlabel('R-squared'); axes[0].set_title('Fit quality')
axes[1].hist(df_fit['fit_kappa_or_width'], bins=60, color='coral')
axes[1].set_xlabel('Kappa'); axes[1].set_title('Tuning sharpness')
axes[2].hist(df_fit['fit_pref_orientation_deg'], bins=36, color='mediumpurple')
axes[2].set_xlabel('Preferred orientation (deg)'); axes[2].set_title('Preferred orientation')
plt.tight_layout(); plt.show()


## Example Fits by Quality

In [ ]:
def von_mises_tuning(theta, b, a, kappa, theta0):
    return b + a * np.exp(kappa * np.cos(2.0 * (theta - theta0)))

rng = np.random.default_rng(42)
theta_fine = np.linspace(0, np.pi, 200)

for label, mask in [('Best (R2>0.9)', df_fit['fit_r2']>0.9),
                    ('Moderate (R2 0.5-0.7)', (df_fit['fit_r2']>=0.5)&(df_fit['fit_r2']<=0.7)),
                    ('Poor (R2<0.3)', df_fit['fit_r2']<0.3)]:
    ids = df_fit[mask].index.values
    if len(ids) < 4: continue
    chosen = rng.choice(ids, size=min(4, len(ids)), replace=False)
    fig, axes = plt.subplots(1, len(chosen), figsize=(4*len(chosen), 3))
    if len(chosen) == 1: axes = [axes]
    for ax, nid in zip(axes, chosen):
        row = df.iloc[nid]
        ax.errorbar(centers_deg, tuning_mean[nid], yerr=tuning_sem[nid], fmt='o', ms=3, capsize=2)
        y_fit = von_mises_tuning(theta_fine, row['fit_baseline'], row['fit_amplitude'],
                                 row['fit_kappa_or_width'], np.radians(row['fit_pref_orientation_deg']))
        ax.plot(np.degrees(theta_fine), y_fit, '-', color='tomato')
        ax.set_title(f'N{nid} R2={row["fit_r2"]:.2f}', fontsize=9)
    fig.suptitle(label); plt.tight_layout(); plt.show()
